In [1]:
!pip install pandas numpy scikit-learn matplotlib seaborn streamlit

Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: C:\Users\YR64S1C\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [2]:
import pandas as pd
import numpy as np
import os
import pickle

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score,
    roc_auc_score,
    precision_score,
    recall_score,
    f1_score,
    matthews_corrcoef,
    confusion_matrix,
    classification_report
)

In [4]:
column_names = [
    "age",
    "workclass",
    "fnlwgt",
    "education",
    "education_num",
    "marital_status",
    "occupation",
    "relationship",
    "race",
    "sex",
    "capital_gain",
    "capital_loss",
    "hours_per_week",
    "native_country",
    "income"
]

train_url = "https://archive.ics.uci.edu/ml/machine-learning-databases/adult/adult.data"
test_url = "https://archive.ics.uci.edu/ml/machine-learning-databases/adult/adult.test"

adult_train = pd.read_csv(
    train_url,
    names=column_names,
    sep=",",
    skipinitialspace=True
)

adult_test = pd.read_csv(
    test_url,
    names=column_names,
    sep=",",
    skipinitialspace=True,
    skiprows=1
)

df = pd.concat(
    [adult_train, adult_test],
    ignore_index=True
)

df.head()

,age,workclass,fnlwgt,education,education_num,marital_status,occupation,relationship,race,sex,capital_gain,capital_loss,hours_per_week,native_country,income
0,39,State-gov,77516,Bachelors,13,Never-married,Adm-clerical,Not-in-family,White,Male,2174,0,40,United-States,<=50K
1,50,Self-emp-not-inc,83311,Bachelors,13,Married-civ-spouse,Exec-managerial,Husband,White,Male,0,0,13,United-States,<=50K
2,38,Private,215646,HS-grad,9,Divorced,Handlers-cleaners,Not-in-family,White,Male,0,0,40,United-States,<=50K
3,53,Private,234721,11th,7,Married-civ-spouse,Handlers-cleaners,Husband,Black,Male,0,0,40,United-States,<=50K
4,28,Private,338409,Bachelors,13,Married-civ-spouse,Prof-specialty,Wife,Black,Female,0,0,40,Cuba,<=50K


In [5]:
for col in df.columns:
    if df[col].dtype == "object":
        df[col] = df[col].str.strip()

df["income"] = df["income"].str.replace(
    ".",
    "",
    regex=False
)

df = df.replace(
    "?",
    np.nan
)

print("Missing values before cleaning:")
print(df.isnull().sum())

df = df.dropna()

print("\nDataset shape after cleaning:")
print(df.shape)

df.head()

Missing values before cleaning:
age                  0
workclass         2799
fnlwgt               0
education            0
education_num        0
marital_status       0
occupation        2809
relationship         0
race                 0
sex                  0
capital_gain         0
capital_loss         0
hours_per_week       0
native_country     857
income               0
dtype: int64

Dataset shape after cleaning:
(45222, 15)


,age,workclass,fnlwgt,education,education_num,marital_status,occupation,relationship,race,sex,capital_gain,capital_loss,hours_per_week,native_country,income
0,39,State-gov,77516,Bachelors,13,Never-married,Adm-clerical,Not-in-family,White,Male,2174,0,40,United-States,<=50K
1,50,Self-emp-not-inc,83311,Bachelors,13,Married-civ-spouse,Exec-managerial,Husband,White,Male,0,0,13,United-States,<=50K
2,38,Private,215646,HS-grad,9,Divorced,Handlers-cleaners,Not-in-family,White,Male,0,0,40,United-States,<=50K
3,53,Private,234721,11th,7,Married-civ-spouse,Handlers-cleaners,Husband,Black,Male,0,0,40,United-States,<=50K
4,28,Private,338409,Bachelors,13,Married-civ-spouse,Prof-specialty,Wife,Black,Female,0,0,40,Cuba,<=50K


In [6]:
df = df.drop(
    columns=[
        "race",
        "sex"
    ]
)

print("Dataset shape after dropping sensitive columns:")
print(df.shape)

print("\nColumns used:")
print(df.columns)

Dataset shape after dropping sensitive columns:
(45222, 13)

Columns used:
Index(['age', 'workclass', 'fnlwgt', 'education', 'education_num',
       'marital_status', 'occupation', 'relationship', 'capital_gain',
       'capital_loss', 'hours_per_week', 'native_country', 'income'],
      dtype='str')


In [7]:
df["income"] = df["income"].map({
    "<=50K": 0,
    ">50K": 1
})

print("Target values:")
print(df["income"].value_counts())

Target values:
income
0    34014
1    11208
Name: count, dtype: int64


In [8]:
df.to_csv(
    "adult_income_dataset.csv",
    index=False
)

print("adult_income_dataset.csv saved successfully")

adult_income_dataset.csv saved successfully


In [9]:
X = df.drop(
    "income",
    axis=1
)

y = df["income"]

print("Feature shape:")
print(X.shape)

print("\nTarget shape:")
print(y.shape)

Feature shape:
(45222, 12)

Target shape:
(45222,)


In [10]:
numerical_columns = X.select_dtypes(
    include=[
        "int64",
        "float64"
    ]
).columns.tolist()

categorical_columns = X.select_dtypes(
    include=[
        "object"
    ]
).columns.tolist()

print("Numerical columns:")
print(numerical_columns)

print("\nCategorical columns:")
print(categorical_columns)

Numerical columns:
['age', 'fnlwgt', 'education_num', 'capital_gain', 'capital_loss', 'hours_per_week']

Categorical columns:
['workclass', 'education', 'marital_status', 'occupation', 'relationship', 'native_country']


C:\Users\YR64S1C\AppData\Local\Temp\ipykernel_3792\2115121767.py:8: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_columns = X.select_dtypes(


In [11]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

test_data = X_test.copy()
test_data["income"] = y_test

test_data.to_csv(
    "test_data.csv",
    index=False
)

print("Training data shape:")
print(X_train.shape)

print("\nTesting data shape:")
print(X_test.shape)

print("\ntest_data.csv saved successfully")

Training data shape:
(36177, 12)

Testing data shape:
(9045, 12)

test_data.csv saved successfully


In [12]:
try:
    encoder = OneHotEncoder(
        handle_unknown="ignore",
        sparse_output=False
    )
except TypeError:
    encoder = OneHotEncoder(
        handle_unknown="ignore",
        sparse=False
    )

preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            StandardScaler(),
            numerical_columns
        ),
        (
            "cat",
            encoder,
            categorical_columns
        )
    ]
)

In [13]:
models = {
    "Logistic Regression": LogisticRegression(
        max_iter=1000,
        solver="liblinear"
    ),

    "Decision Tree": DecisionTreeClassifier(
        random_state=42,
        max_depth=10
    ),

    "kNN": KNeighborsClassifier(
        n_neighbors=7,
        n_jobs=-1
    ),

    "Naive Bayes": GaussianNB(),

    "Random Forest": RandomForestClassifier(
        n_estimators=150,
        random_state=42,
        n_jobs=-1
    )
}

In [14]:
os.makedirs(
    "models",
    exist_ok=True
)

results = []

for model_name, model in models.items():

    print("Training:", model_name)

    pipeline = Pipeline(
        steps=[
            (
                "preprocessor",
                preprocessor
            ),
            (
                "classifier",
                model
            )
        ]
    )

    pipeline.fit(
        X_train,
        y_train
    )

    y_pred = pipeline.predict(
        X_test
    )

    y_prob = pipeline.predict_proba(
        X_test
    )[:, 1]

    accuracy = accuracy_score(
        y_test,
        y_pred
    )

    auc = roc_auc_score(
        y_test,
        y_prob
    )

    precision = precision_score(
        y_test,
        y_pred
    )

    recall = recall_score(
        y_test,
        y_pred
    )

    f1 = f1_score(
        y_test,
        y_pred
    )

    mcc = matthews_corrcoef(
        y_test,
        y_pred
    )

    results.append([
        model_name,
        accuracy,
        auc,
        precision,
        recall,
        f1,
        mcc
    ])

    model_file_name = model_name.lower().replace(
        " ",
        "_"
    ) + ".pkl"

    model_path = os.path.join(
        "models",
        model_file_name
    )

    with open(
        model_path,
        "wb"
    ) as file:
        pickle.dump(
            pipeline,
            file
        )

    print(model_name, "completed and saved")

print("\nAll models trained successfully")

Training: Logistic Regression
Logistic Regression completed and saved
Training: Decision Tree
Decision Tree completed and saved
Training: kNN
kNN completed and saved
Training: Naive Bayes
Naive Bayes completed and saved
Training: Random Forest
Random Forest completed and saved

All models trained successfully


In [15]:
result_df = pd.DataFrame(
    results,
    columns=[
        "Model",
        "Accuracy",
        "AUC",
        "Precision",
        "Recall",
        "F1",
        "MCC"
    ]
)

result_df = result_df.round(4)

result_df

,Model,Accuracy,AUC,Precision,Recall,F1,MCC
0,Logistic Regression,0.8446,0.9013,0.7333,0.5861,0.6515,0.5588
1,Decision Tree,0.8482,0.8987,0.7820,0.5375,0.6371,0.5605
2,kNN,0.8329,0.8749,0.6847,0.6044,0.6420,0.5354
3,Naive Bayes,0.6035,0.8450,0.3781,0.9300,0.5376,0.3770
4,Random Forest,0.8493,0.9029,0.7310,0.6204,0.6712,0.5775


In [16]:
result_df.to_csv(
    "model_results.csv",
    index=False
)

print("model_results.csv saved successfully")

model_results.csv saved successfully


In [17]:
best_model = result_df.sort_values(
    by=[
        "AUC",
        "Accuracy"
    ],
    ascending=False
).iloc[0]

print("Best Performing Model:")
print(best_model)

Best Performing Model:
Model        Random Forest
Accuracy            0.8493
AUC                 0.9029
Precision            0.731
Recall              0.6204
F1                  0.6712
MCC                 0.5775
Name: 4, dtype: object


In [18]:
requirements_content = """streamlit
scikit-learn
numpy
pandas
matplotlib
seaborn
"""

with open(
    "requirements.txt",
    "w"
) as file:
    file.write(
        requirements_content
    )

print("requirements.txt created successfully")

requirements.txt created successfully


In [19]:
app_code = '''
import streamlit as st
import pandas as pd
import pickle
import os

from sklearn.metrics import (
    accuracy_score,
    roc_auc_score,
    precision_score,
    recall_score,
    f1_score,
    matthews_corrcoef,
    confusion_matrix,
    classification_report
)

st.set_page_config(
    page_title="Adult Income Classification App",
    layout="wide"
)

st.title("Machine Learning Assignment 2")
st.subheader("Adult Income Classification using Machine Learning")

st.write(
    "This Streamlit application predicts whether annual income is above 50K using the Adult Income Dataset."
)

model_options = {
    "Logistic Regression": "models/logistic_regression.pkl",
    "Decision Tree": "models/decision_tree.pkl",
    "kNN": "models/knn.pkl",
    "Naive Bayes": "models/naive_bayes.pkl",
    "Random Forest": "models/random_forest.pkl"
}

selected_model = st.selectbox(
    "Select Machine Learning Model",
    list(model_options.keys())
)

uploaded_file = st.file_uploader(
    "Upload test_data.csv file",
    type=["csv"]
)

if uploaded_file is not None:
    data = pd.read_csv(uploaded_file)
    st.success("File uploaded successfully")
else:
    if os.path.exists("test_data.csv"):
        data = pd.read_csv("test_data.csv")
        st.info("Using default test_data.csv from repository")
    else:
        st.warning("Please upload test_data.csv")
        st.stop()

st.write("Dataset Preview")
st.dataframe(
    data.head(),
    use_container_width=True
)

if "income" not in data.columns:
    st.error("The uploaded file must contain the target column named income")
    st.stop()

X_test = data.drop(
    "income",
    axis=1
)

y_test = data["income"]

model_path = model_options[selected_model]

with open(
    model_path,
    "rb"
) as file:
    model = pickle.load(file)

y_pred = model.predict(
    X_test
)

y_prob = model.predict_proba(
    X_test
)[:, 1]

accuracy = accuracy_score(
    y_test,
    y_pred
)

auc = roc_auc_score(
    y_test,
    y_prob
)

precision = precision_score(
    y_test,
    y_pred
)

recall = recall_score(
    y_test,
    y_pred
)

f1 = f1_score(
    y_test,
    y_pred
)

mcc = matthews_corrcoef(
    y_test,
    y_pred
)

st.write("Evaluation Metrics")

col1, col2, col3 = st.columns(3)
col4, col5, col6 = st.columns(3)

col1.metric("Accuracy", round(accuracy, 4))
col2.metric("AUC", round(auc, 4))
col3.metric("Precision", round(precision, 4))
col4.metric("Recall", round(recall, 4))
col5.metric("F1 Score", round(f1, 4))
col6.metric("MCC Score", round(mcc, 4))

st.write("Confusion Matrix")

cm = confusion_matrix(
    y_test,
    y_pred
)

st.write(cm)

st.write("Classification Report")

report = classification_report(
    y_test,
    y_pred,
    output_dict=True
)

st.dataframe(
    pd.DataFrame(report).transpose(),
    use_container_width=True
)

prediction_output = data.copy()
prediction_output["Predicted Income Class"] = y_pred

st.write("Prediction Output")
st.dataframe(
    prediction_output.head(20),
    use_container_width=True
)

csv_output = prediction_output.to_csv(
    index=False
).encode("utf-8")

st.download_button(
    label="Download Predictions",
    data=csv_output,
    file_name="adult_income_predictions.csv",
    mime="text/csv"
)
'''

with open(
    "app.py",
    "w"
) as file:
    file.write(
        app_code
    )

print("app.py created successfully")

app.py created successfully


_IncompleteInputError: incomplete input (1541669725.py, line 87)